# Lesson 03 Lab — GPU Memory as a Spatial Hierarchy

**Puzzle:** Why can the same byte be cheap in a register and expensive in external memory even when its numerical value never changes?

This notebook retains one complete RTX 5090 execution.


## Why this matters

GPU memory names also describe physical placement and sharing scope. Registers and shared memory sit inside an SM; L2 is shared across SMs on the GPU die; HBM or GDDR is outside the die and reached through controllers and physical links. Capacity tends to increase outward while latency, energy, and sharing distance also increase. CUDA's address spaces are a programming interface over this physical hierarchy, not a one-to-one schematic.


## 0. Predict before running

1. Place registers, shared memory, L2, and HBM/GDDR from nearest to farthest from an SM.
2. Predict the outer-memory bytes for reuse counts 1 and 32.
3. Explain why CUDA local memory is not necessarily on-chip.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The lesson builds an explicit hierarchy record with capacity, modeled latency, scope, and technology, then evaluates the same working set under different reuse assumptions. A byte fetched once from external memory and reused many times on chip amortizes the outer transfer. A byte streamed once does not. The model therefore records both distance and reuse instead of ranking memories by a single latency number.

- Registers/shared memory, L2, and external memory occupy different physical regions.
- Capacity, scope, latency, and bandwidth are separate axes.
- Reuse changes how often the expensive outer path is paid.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["registers"] --> B["shared memory / L1"]
  B --> C["L2 slices"]
  C --> D["memory controllers"]
  D --> E["HBM or GDDR"]
```


## 3. Inspect the visual boundary

![GPU memory spatial layout](../assets/visualizations/gpu-memory-spatial-layout.png)

- [Interactive memory layout](../assets/visualizations/gpu-memory-spatial-layout.html)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 3
LESSON_TITLE = 'GPU Memory as a Spatial Hierarchy'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260816
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | one-pass streaming of a fixed working set |
| Candidate | the same working set staged once and reused on chip |
| Held constant | working-set bytes and hierarchy assumptions |
| Measurements | modeled access ratio and external bytes per use |
| Evidence | `capacity-model` |

**Experiment:** Evaluate an explicit hierarchy and amortize external traffic over reuse.


## 6. Inspect the code

The code keeps the hierarchy as data rather than hiding it in prose. It computes bytes per logical use for several reuse counts and prints the actual CUDA device memory capacity as an environment fact.

Do not run until the code matches the frozen table.


In [2]:
hierarchy = [
    {"level": "register", "scope": "thread", "technology": "flip-flop/register file", "latency_ratio": 1},
    {"level": "shared/L1", "scope": "block/SM", "technology": "SRAM", "latency_ratio": 5},
    {"level": "L2", "scope": "all SMs", "technology": "SRAM", "latency_ratio": 25},
    {"level": "external memory", "scope": "device", "technology": "GDDR7 on this GPU", "latency_ratio": 120},
]
working_set = 64 * 2**20
reuse_counts = (1, 2, 4, 8, 16, 32)
bytes_per_use = {str(r): working_set / r for r in reuse_counts}
device_gib = props.total_memory / 2**30
metrics = {
    "hierarchy": hierarchy,
    "working_set_bytes": working_set,
    "device_memory_gib": device_gib,
    "bytes_per_use": bytes_per_use,
    "bytes_per_use_streaming": int(bytes_per_use["1"]),
    "bytes_per_use_reuse32": int(bytes_per_use["32"]),
    "reuse32_reduction": bytes_per_use["1"] / bytes_per_use["32"],
}
analysis = (
    f"A {working_set / 2**20:.0f} MiB working set costs {working_set:,} external bytes per use "
    f"when streamed once, but {bytes_per_use['32']:,.0f} bytes per logical use when one load is "
    "amortized across 32 on-chip uses. Latency ratios are illustrative."
)
print(json.dumps(metrics, indent=2))


{
  "hierarchy": [
    {
      "level": "register",
      "scope": "thread",
      "technology": "flip-flop/register file",
      "latency_ratio": 1
    },
    {
      "level": "shared/L1",
      "scope": "block/SM",
      "technology": "SRAM",
      "latency_ratio": 5
    },
    {
      "level": "L2",
      "scope": "all SMs",
      "technology": "SRAM",
      "latency_ratio": 25
    },
    {
      "level": "external memory",
      "scope": "device",
      "technology": "GDDR7 on this GPU",
      "latency_ratio": 120
    }
  ],
  "working_set_bytes": 67108864,
  "device_memory_gib": 31.35833740234375,
  "bytes_per_use": {
    "1": 67108864.0,
    "2": 33554432.0,
    "4": 16777216.0,
    "8": 8388608.0,
    "16": 4194304.0,
    "32": 2097152.0
  },
  "bytes_per_use_streaming": 67108864,
  "bytes_per_use_reuse32": 2097152,
  "reuse32_reduction": 32.0
}


## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Device memory | 31.3583 |
| Streaming bytes/use | 67,108,864 bytes |
| 32× reuse bytes/use | 2,097,152 bytes |
| Traffic reduction | 32.000x |


## 8. Explain rather than overclaim

A 64 MiB working set costs 67,108,864 external bytes per use when streamed once, but 2,097,152 bytes per logical use when one load is amortized across 32 on-chip uses. Latency ratios are illustrative.

**Evidence boundary:** Measured environment facts feed explicit capacity or Roofline arithmetic. Declared hierarchy and resource fields remain assumptions until native counters confirm them.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 3, "title": 'GPU Memory as a Spatial Hierarchy', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": "Optimize placement only after naming scope and reuse; 'faster memory' without a tile lifetime and sharing contract is incomplete.",
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 3,
  "title": "GPU Memory as a Spatial Hierarchy",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260816
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "hierarchy": [
      {
        "level": "register",
        "scope": "thread",
        "technology": "flip-flop/register file",
        "latency_ratio": 1
      },
      {
        "level": "shared/L1",
        "scope": "block/SM",
        "technology": "SRAM",
        "latency_ratio": 5
      },
      {
        "level": "L2",
        "scope": "all SMs",
        "technology": "SRAM",
        "latency_ratio": 25
      },
      {
        "level": "external memory",
        "scope": "device",
        "technology": "GDDR7 on this GPU",
        "latency_ratio": 120
      }
    ],
    "working_set_bytes": 67108864,
    "device_memory_gib": 31.35833740234375,
    "bytes_per_use":

## 10. Make the decision

> Optimize placement only after naming scope and reuse; 'faster memory' without a tile lifetime and sharing contract is incomplete.

**Failure analysis:** Latency values are educational ratios, not microbenchmarks. Cache replacement, compiler decisions, occupancy, and contention can change the realized path.


## 11. Extend the evidence

Use Nsight Compute counters to measure DRAM, L2, and L1/shared traffic for a tiled kernel and compare the measured reuse to the model.

See [`README.md`](README.md) for the full explanation and references.
